## 📦 Installation

In [1]:
# Install required packages
!pip install -q sentence-transformers faiss-cpu FlagEmbedding rank-bm25

## 📚 Imports

In [1]:
# Core Libraries
import os
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from typing import List, Dict, Tuple
import re

# Embedding & Retrieval
from sentence_transformers import SentenceTransformer
import faiss

# Reranking
from FlagEmbedding import FlagReranker

# BM25
from rank_bm25 import BM25Okapi

# Utils
import warnings
warnings.filterwarnings('ignore')
import logging
logging.disable(logging.CRITICAL)

In [2]:
# Check GPU
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    device = 'cuda'
else:
    print("Running on CPU")
    device = 'cpu'

CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


## ⚙️ Configuration

In [3]:
# Load configuration from config.py
from config import CONFIG, print_config

# Print configuration summary
print_config()


⚙️ QUICK WINS CONFIGURATION

📂 Data Directory: ..\..\data
📄 Output File: submission_improved.csv

🤖 Models:
   Embedding: intfloat/e5-small-v2
   Reranker: BAAI/bge-reranker-v2-m3

✂️ Chunking:
   use_chunking: True
   chunk_size: 512
   chunk_overlap: 128
   chunk_aggregation: max
   preserve_tables: True

🔍 Hybrid Search:
   use_hybrid: True
   hybrid_alpha: 0.6

🎯 Retrieval:
   top_k_retrieval: 100
   top_k_rerank: 50
   top_k_final: 10
   embed_batch_size: 16
   max_length: 4096
   eval_on_qrels: True


## 🔧 Helper Functions

In [4]:
# Import shared utilities
import sys
sys.path.insert(0, '..')
from utils import (
    load_jsonl_data,
    QRELS_MAPPING,
    DATASETS
)

print("✅ Data loading functions imported from utils.py")

✅ Data loading functions imported from utils.py


In [5]:
def detect_tables(text: str) -> List[Tuple[int, int]]:
    """Detect table regions using heuristics"""
    lines = text.split('\n')
    table_regions = []
    in_table = False
    table_start = 0
    
    for i, line in enumerate(lines):
        is_table = (
            line.count('|') >= 2 or
            line.count('\t') >= 2 or
            len(re.findall(r'\s{3,}', line)) >= 2
        )
        
        if is_table and not in_table:
            in_table = True
            table_start = max(0, i - 1)
        elif not is_table and in_table:
            in_table = False
            table_end = min(len(lines), i + 1)
            if table_end - table_start >= 3:
                table_regions.append((table_start, table_end))
    
    if in_table:
        table_regions.append((table_start, len(lines)))
    
    return table_regions


def chunk_text_simple(text: str, chunk_size: int, overlap: int) -> List[str]:
    """Sliding window chunking"""
    words = text.split()
    if len(words) <= chunk_size:
        return [text]
    
    chunks = []
    step = chunk_size - overlap
    for i in range(0, len(words), step):
        chunk = ' '.join(words[i:i + chunk_size])
        if len(words[i:i + chunk_size]) >= 50:
            chunks.append(chunk)
    return chunks


def chunk_document_smart(doc_id: str, title: str, text: str, 
                        chunk_size: int, overlap: int, preserve_tables: bool):
    """Smart chunking with table preservation"""
    chunks = []
    
    if not preserve_tables or len(text) < 500:
        for i, chunk_text in enumerate(chunk_text_simple(text, chunk_size, overlap)):
            chunks.append({
                'chunk_id': f"{doc_id}_c{i}",
                'text': f"[{title}] {chunk_text}",
                'doc_id': doc_id
            })
        return chunks
    
    lines = text.split('\n')
    table_regions = detect_tables(text)
    
    if not table_regions:
        for i, chunk_text in enumerate(chunk_text_simple(text, chunk_size, overlap)):
            chunks.append({
                'chunk_id': f"{doc_id}_c{i}",
                'text': f"[{title}] {chunk_text}",
                'doc_id': doc_id
            })
        return chunks
    
    chunk_idx = 0
    prev_end = 0
    
    for table_start, table_end in table_regions:
        # Text before table
        if table_start > prev_end:
            before = '\n'.join(lines[prev_end:table_start])
            if before.strip():
                for chunk_text in chunk_text_simple(before, chunk_size, overlap):
                    chunks.append({
                        'chunk_id': f"{doc_id}_c{chunk_idx}",
                        'text': f"[{title}] {chunk_text}",
                        'doc_id': doc_id
                    })
                    chunk_idx += 1
        
        # Table as single chunk
        table_text = '\n'.join(lines[table_start:table_end])
        chunks.append({
            'chunk_id': f"{doc_id}_t{chunk_idx}",
            'text': f"[TABLE from {title}]\n{table_text}",
            'doc_id': doc_id
        })
        chunk_idx += 1
        prev_end = table_end
    
    # Text after table
    if prev_end < len(lines):
        after = '\n'.join(lines[prev_end:])
        if after.strip():
            for chunk_text in chunk_text_simple(after, chunk_size, overlap):
                chunks.append({
                    'chunk_id': f"{doc_id}_c{chunk_idx}",
                    'text': f"[{title}] {chunk_text}",
                    'doc_id': doc_id
                })
                chunk_idx += 1
    
    return chunks

In [6]:
# Import retrieval utilities from shared utils
from utils import normalize_scores, hybrid_search, aggregate_chunk_scores

print("✅ Retrieval functions imported from utils.py")

✅ Retrieval functions imported from utils.py


In [7]:
# Import evaluation functions from shared utils
from utils import compute_ndcg_batch as compute_ndcg, evaluate_results_df as evaluate_results

print("✅ Evaluation functions imported from utils.py")

✅ Evaluation functions imported from utils.py


## 🤖 Load Models

In [14]:
print("Loading models...")

# Embedding
print(f"\n1. Loading: {CONFIG['embedding_model']}")
embed_model = SentenceTransformer(CONFIG['embedding_model'], device=device)
print("   ✅ Done")

# Reranker
print(f"\n2. Loading: {CONFIG['reranker_model']}")
reranker = FlagReranker(CONFIG['reranker_model'], use_fp16=(device=='cuda'))
print("   ✅ Done")

print("\n✅ All models loaded!")

Loading models...

1. Loading: intfloat/e5-small-v2
   ✅ Done

2. Loading: BAAI/bge-reranker-v2-m3
   ✅ Done

✅ All models loaded!


## 🔄 Main Pipeline

In [15]:
def process_dataset_improved(dataset_name: str, config: Dict):
    """Improved pipeline"""
    print(f"\n{'='*60}")
    print(f"Processing: {dataset_name.upper()}")
    print(f"{'='*60}")
    
    # Load
    corpus_df, queries_df, qrels_df = load_jsonl_data(dataset_name, config['data_dir'])
    
    # Chunk
    print(f"\n📄 Chunking...")
    all_chunks = []
    chunk_to_doc = {}
    
    if config['use_chunking']:
        for _, row in tqdm(corpus_df.iterrows(), total=len(corpus_df), desc="Chunk"):
            chunks = chunk_document_smart(
                row['_id'], str(row.get('title', '')), str(row.get('text', '')),
                config['chunk_size'], config['chunk_overlap'], config['preserve_tables']
            )
            for c in chunks:
                all_chunks.append(c)
                chunk_to_doc[c['chunk_id']] = c['doc_id']
        print(f"   {len(all_chunks)} chunks from {len(corpus_df)} docs")
    else:
        for _, row in corpus_df.iterrows():
            doc_id = row['_id']
            text = f"[{row.get('title', '')}] {row.get('text', '')}"
            all_chunks.append({'chunk_id': doc_id, 'text': text, 'doc_id': doc_id})
            chunk_to_doc[doc_id] = doc_id
    
    chunk_texts = [c['text'] for c in all_chunks]
    chunk_ids = [c['chunk_id'] for c in all_chunks]
    
    # Embed
    print(f"\n🔢 Embedding...")
    chunk_embeddings = embed_model.encode(
        chunk_texts, batch_size=config['embed_batch_size'],
        show_progress_bar=True, convert_to_numpy=True,
        normalize_embeddings=True
    )
    
    # FAISS
    print(f"\n🔍 Building FAISS...")
    index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
    index.add(chunk_embeddings.astype('float32'))
    
    # BM25
    bm25 = None
    if config['use_hybrid']:
        print(f"\n🔤 Building BM25...")
        tokenized = [t.lower().split() for t in chunk_texts]
        bm25 = BM25Okapi(tokenized)
    
    del chunk_embeddings
    if device == 'cuda':
        torch.cuda.empty_cache()
    
    # Queries
    print(f"\n🎯 Processing queries...")
    query_texts = [str(r.get('text', '')) for _, r in queries_df.iterrows()]
    query_ids = queries_df['_id'].tolist()
    
    query_embeddings = embed_model.encode(
        query_texts, batch_size=config['embed_batch_size'],
        show_progress_bar=True, convert_to_numpy=True,
        normalize_embeddings=True
    )
    
    # Retrieve
    results = []
    for i, query_id in enumerate(tqdm(query_ids, desc="Retrieve+Rerank")):
        query_emb = query_embeddings[i]
        query_text = query_texts[i]
        
        # Hybrid or dense
        if config['use_hybrid'] and bm25:
            scores, chunk_indices = hybrid_search(
                query_emb, query_text, index, bm25, chunk_texts,
                config['top_k_retrieval'], config['hybrid_alpha']
            )
        else:
            scores, chunk_indices = index.search(
                query_emb.reshape(1, -1).astype('float32'),
                config['top_k_retrieval']
            )
            scores, chunk_indices = scores[0], chunk_indices[0]
        
        # Aggregate to docs
        doc_scores = {}
        for idx, score in zip(chunk_indices, scores):
            doc_id = chunk_to_doc[chunk_ids[idx]]
            if doc_id not in doc_scores:
                doc_scores[doc_id] = []
            doc_scores[doc_id].append(float(score))
        
        doc_agg = aggregate_chunk_scores(doc_scores, config['chunk_aggregation'])
        sorted_docs = sorted(doc_agg.items(), key=lambda x: x[1], reverse=True)[:config['top_k_rerank']]
        
        # Rerank
        candidate_ids = [d[0] for d in sorted_docs]
        candidate_texts = [
            str(corpus_df[corpus_df['_id']==d]['text'].values[0])[:2048]
            for d in candidate_ids
        ]
        
        pairs = [[query_text, t] for t in candidate_texts]
        rerank_scores = reranker.compute_score(pairs)
        
        if not isinstance(rerank_scores, list):
            rerank_scores = [rerank_scores]
        
        scored = list(zip(candidate_ids, rerank_scores))
        scored.sort(key=lambda x: x[1], reverse=True)
        
        for doc_id, score in scored[:config['top_k_final']]:
            results.append({
                'query_id': query_id,
                'corpus_id': doc_id,
                'score': float(score)
            })
    
    results_df = pd.DataFrame(results)
    print(f"\n✅ Done: {len(results_df)} results")
    
    # Evaluate
    eval_metrics = {}
    if config['eval_on_qrels'] and qrels_df is not None:
        print(f"\n📊 Evaluating...")
        eval_metrics = evaluate_results(results_df, qrels_df)
        print(f"   NDCG@10: {eval_metrics['NDCG@10']:.4f}")
    
    del query_embeddings, index
    if device == 'cuda':
        torch.cuda.empty_cache()
    
    return results_df, eval_metrics

## 🚀 Run Pipeline

In [16]:
all_results = []
all_eval = {}
failed = []

for dataset in CONFIG['datasets']:
    try:
        df_res, metrics = process_dataset_improved(dataset, CONFIG)
        all_results.append(df_res)
        if metrics:
            all_eval[dataset] = metrics
    except Exception as e:
        print(f"\n❌ Error: {dataset}: {e}")
        import traceback
        traceback.print_exc()
        failed.append(dataset)

print(f"\n{'='*60}")
print(f"✅ Done: {len(all_results)}/{len(CONFIG['datasets'])}")
if failed:
    print(f"❌ Failed: {failed}")


Processing: CONVFINQA
  Loaded 2066 docs, 421 queries

📄 Chunking...


Chunk:   0%|          | 0/2066 [00:00<?, ?it/s]

   7473 chunks from 2066 docs

🔢 Embedding...


Batches:   0%|          | 0/468 [00:00<?, ?it/s]


🔍 Building FAISS...

🔤 Building BM25...

🎯 Processing queries...


...

Batches:   0%|          | 0/360 [00:00<?, ?it/s]


🔍 Building FAISS...

🔤 Building BM25...

🎯 Processing queries...


Batches:   0%|          | 0/104 [00:00<?, ?it/s]

Retrieve+Rerank:   0%|          | 0/1663 [00:00<?, ?it/s]


✅ Done: 16630 results

📊 Evaluating...
   NDCG@10: 0.4999

✅ Done: 7/7


## 📊 Evaluation Summary

In [17]:
if all_eval:
    print("\n📊 Local Evaluation (30% qrels):")
    print("="*60)
    
    total_ndcg = 0
    total_queries = 0
    
    for ds, m in all_eval.items():
        print(f"\n{ds.upper()}: NDCG@10 = {m['NDCG@10']:.4f}")
        total_ndcg += m['NDCG@10'] * m['num_qrels']
        total_queries += m['num_qrels']
    
    if total_queries > 0:
        avg_ndcg = total_ndcg / total_queries
        print(f"\n{'='*60}")
        print(f"📈 AVERAGE NDCG@10: {avg_ndcg:.4f}")
        print(f"{'='*60}")
        
        baseline = 0.328
        gain = avg_ndcg - baseline
        gain_pct = (gain / baseline) * 100
        
        print(f"\n🎯 vs Baseline:")
        print(f"   Baseline: {baseline:.4f}")
        print(f"   Improved: {avg_ndcg:.4f}")
        print(f"   Gain: +{gain:.4f} ({gain_pct:+.1f}%)")
        
        if avg_ndcg >= 0.58:
            print(f"\n🏆 Likely TOP 3!")
        elif avg_ndcg >= 0.50:
            print(f"\n✅ Good progress, tune more!")
        else:
            print(f"\n⚠️ Need more work")


📊 Local Evaluation (30% qrels):

CONVFINQA: NDCG@10 = 0.4756

FINANCEBENCH: NDCG@10 = 0.3521

FINDER: NDCG@10 = 0.3393

FINQA: NDCG@10 = 0.4334

FINQABENCH: NDCG@10 = 0.8662

MULTIHEIRTT: NDCG@10 = 0.1553

TATQA: NDCG@10 = 0.4999

📈 AVERAGE NDCG@10: 0.4052

🎯 vs Baseline:
   Baseline: 0.3280
   Improved: 0.4052
   Gain: +0.0772 (+23.5%)

⚠️ Need more work


## 💾 Generate Submission

In [18]:
if all_results:
    final_df = pd.concat(all_results, ignore_index=True)
    submission_df = final_df[['query_id', 'corpus_id']]
    
    submission_df.to_csv(CONFIG['output_file'], index=False)
    
    print(f"\n✅ Saved: {CONFIG['output_file']}")
    print(f"   Entries: {len(submission_df)}")
    print(f"   Queries: {submission_df['query_id'].nunique()}")
    
    print(f"\n📋 Sample:")
    print(submission_df.head(10))
    
    counts = submission_df.groupby('query_id').size()
    print(f"\n🔍 Validation:")
    print(f"   Per query: {counts.value_counts().to_dict()}")
    if (counts == 10).all():
        print(f"   ✅ All queries have 10 results")
else:
    print("\n❌ No results")


✅ Saved: submission_improved.csv
   Entries: 46710
   Queries: 4671

📋 Sample:
    query_id  corpus_id
0  qd4982518  dd4c4f7aa
1  qd4982518  dd4bb016e
2  qd4982518  dd4b9f7f6
3  qd4982518  dd4bb5506
4  qd4982518  dd4b87d18
5  qd4982518  dd4be45d6
6  qd4982518  dd4bd3790
7  qd4982518  dd4bad4e6
8  qd4982518  dd4b89cbc
9  qd4982518  dd4baed28

🔍 Validation:
   Per query: {10: 4671}
   ✅ All queries have 10 results


## 🎯 Summary

In [ ]:
print("\n" + "="*60)
print("🎉 IMPROVED PIPELINE COMPLETED!")
print("="*60)

print("\n✅ Improvements:")
print("   1. Table-aware chunking")
print("   2. BGE-reranker-v2-m3 (SOTA)")
print("   3. Hybrid retrieval (BM25+Dense)")
print("   4. Local evaluation")
print("   5. Optimized parameters")

print("\n💾 Next: Submit to Kaggle!")
print("="*60)


🎉 IMPROVED PIPELINE COMPLETED!

✅ Improvements:
   1. Table-aware chunking
   2. BGE-reranker-v2-m3 (SOTA)
   3. Hybrid retrieval (BM25+Dense)
   4. Local evaluation
   5. Optimized parameters

💾 Next: Submit to Kaggle!


: 